# Ocean Bootstrap Analysis

Read `results/ocean/ocean_model_comparison_bootstrap_best.csv` and produce one simple table: average metric values over bootstrap seeds for each model.


In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda value: f'{value:,.6g}')


## Load Results


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists() and (path / 'results' / 'ocean').exists():
            return path
    raise FileNotFoundError('Could not find the repository root from the current working directory.')


PROJECT_ROOT = find_project_root()
RESULTS_PATH = PROJECT_ROOT / 'results' / 'ocean' / 'ocean_model_comparison_bootstrap_best.csv'

df = pd.read_csv(RESULTS_PATH)
if df.empty:
    raise ValueError(f'No rows found in {RESULTS_PATH}')

df['K'] = df['K'].astype('Int64')
df['L'] = df['L'].astype('Int64')
df['size_label'] = 'K=' + df['K'].astype(str)
df.loc[df['L'].notna(), 'size_label'] += ', L=' + df.loc[df['L'].notna(), 'L'].astype(str)

run_overview = pd.DataFrame([
    {
        'results_path': str(RESULTS_PATH.relative_to(PROJECT_ROOT)),
        'rows': len(df),
        'bootstrap_seeds': df['split_seed'].nunique(),
        'candidate_models': df['candidate_id'].nunique(),
        'rows_per_seed_min': df.groupby('split_seed').size().min(),
        'rows_per_seed_max': df.groupby('split_seed').size().max(),
    }
])
run_overview


,results_path,rows,bootstrap_seeds,candidate_models,rows_per_seed_min,rows_per_seed_max
0,results/ocean/ocean_model_comparison_bootstrap...,800,100,8,8,8


## Average Metrics by Model


In [3]:
METRIC_COLUMNS = [
    'success',
    'converged',
    'train_bic',
    'train_aic',
    'heldout_bic',
    'heldout_aic',
    'train_log_likelihood',
    'heldout_log_likelihood',
    'train_avg_log_likelihood',
    'heldout_avg_log_likelihood',
    'train_gmpd',
    'heldout_gmpd',
    'n_free_params',
    'n_iter',
    'fit_time',
]

MODEL_COLUMNS = [
    'candidate_id',
    'display_name',
    'candidate_model_type',
    'candidate_family',
    'K',
    'L',
    'size_label',
]

model_metric_averages = (
    df.groupby(MODEL_COLUMNS, dropna=False, observed=True)
    .agg(
        n_seeds=('split_seed', 'nunique'),
        **{f'{column}_mean': (column, 'mean') for column in METRIC_COLUMNS},
    )
    .reset_index()
    .sort_values('heldout_avg_log_likelihood_mean', ascending=False)
)

model_metric_averages


,candidate_id,display_name,candidate_model_type,candidate_family,K,L,size_label,n_seeds,success_mean,converged_mean,train_bic_mean,train_aic_mean,heldout_bic_mean,heldout_aic_mean,train_log_likelihood_mean,heldout_log_likelihood_mean,train_avg_log_likelihood_mean,heldout_avg_log_likelihood_mean,train_gmpd_mean,heldout_gmpd_mean,n_free_params_mean,n_iter_mean,fit_time_mean
7,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",two_layer_mixture,vmf_gaussian,6,4,"K=6, L=4",100,1,1,1.12094e+06,1.11845e+06,"282,509","280,368","-558,973","-139,933",-3.73166,-3.73663,0.0239532,0.0238356,251,61.33,72.0955
3,gauss_vmf_tl_6x4,"Gaussian/vMF two-layer, (K,L)=(6,4)",two_layer_mixture,gaussian_vmf,6,4,"K=6, L=4",100,1,1,1.1309e+06,1.12967e+06,"283,947","282,880","-564,708","-141,315",-3.76994,-3.77354,0.0230534,0.0229723,125,48.43,55.2199
6,vmf_gauss_tl_3x2,"vMF/Gaussian two-layer, (K,L)=(3,2)",two_layer_mixture,vmf_gaussian,3,2,"K=3, L=2",100,1,1,1.14167e+06,1.14103e+06,"286,176","285,622","-570,449","-142,746",-3.80827,-3.81174,0.0221865,0.0221115,65,24.22,11.5122
1,full_cyl_k6,"Full cylindrical mixture, K=6",cylindrical_mixture,full,6,<NA>,K=6,100,1,1,1.14293e+06,1.14187e+06,"286,752","285,839","-570,828","-142,812",-3.8108,-3.81352,0.0221308,0.0220723,107,58.45,24.802
5,ind_cyl_k6,"Independent cylindrical mixture, K=6",cylindrical_mixture,independent,6,<NA>,K=6,100,1,1,1.14718e+06,1.14647e+06,"287,544","286,938","-573,165","-143,398",-3.82641,-3.82916,0.0217879,0.0217294,71,63.59,23.5868
0,full_cyl_k3,"Full cylindrical mixture, K=3",cylindrical_mixture,full,3,<NA>,K=3,100,1,1,1.1669e+06,1.16637e+06,"292,319","291,867","-583,132","-145,880",-3.89295,-3.89544,0.0203852,0.0203356,53,25.45,8.61645
2,gauss_vmf_tl_3x2,"Gaussian/vMF two-layer, (K,L)=(3,2)",two_layer_mixture,gaussian_vmf,3,2,"K=3, L=2",100,1,1,1.17015e+06,1.16972e+06,"293,036","292,660","-584,815","-146,286",-3.90418,-3.90627,0.0201575,0.0201164,44,57.88,24.6919
4,ind_cyl_k3,"Independent cylindrical mixture, K=3",cylindrical_mixture,independent,3,<NA>,K=3,100,1,1,1.18417e+06,1.18382e+06,"296,469","296,170","-591,877","-148,050",-3.95132,-3.95338,0.0192293,0.019191,35,33.71,9.16741


In [7]:
model_metric_averages[
    model_metric_averages['K'] == 3
][
    ['display_name',
    'n_free_params_mean',
    'train_bic_mean',
    'train_aic_mean',
    'heldout_avg_log_likelihood_mean',
    'n_iter_mean',
    'fit_time_mean'
    ]
]

,display_name,n_free_params_mean,train_bic_mean,train_aic_mean,heldout_avg_log_likelihood_mean,n_iter_mean,fit_time_mean
6,"vMF/Gaussian two-layer, (K,L)=(3,2)",65,1.14167e+06,1.14103e+06,-3.81174,24.22,11.5122
0,"Full cylindrical mixture, K=3",53,1.1669e+06,1.16637e+06,-3.89544,25.45,8.61645
2,"Gaussian/vMF two-layer, (K,L)=(3,2)",44,1.17015e+06,1.16972e+06,-3.90627,57.88,24.6919
4,"Independent cylindrical mixture, K=3",35,1.18417e+06,1.18382e+06,-3.95338,33.71,9.16741


In [8]:
model_metric_averages[
    model_metric_averages['K'] == 6
][
    ['display_name',
    'n_free_params_mean',
    'train_bic_mean',
    'train_aic_mean',
    'heldout_avg_log_likelihood_mean',
    'n_iter_mean',
    'fit_time_mean'
    ]
]

,display_name,n_free_params_mean,train_bic_mean,train_aic_mean,heldout_avg_log_likelihood_mean,n_iter_mean,fit_time_mean
7,"vMF/Gaussian two-layer, (K,L)=(6,4)",251,1.12094e+06,1.11845e+06,-3.73663,61.33,72.0955
3,"Gaussian/vMF two-layer, (K,L)=(6,4)",125,1.1309e+06,1.12967e+06,-3.77354,48.43,55.2199
1,"Full cylindrical mixture, K=6",107,1.14293e+06,1.14187e+06,-3.81352,58.45,24.802
5,"Independent cylindrical mixture, K=6",71,1.14718e+06,1.14647e+06,-3.82916,63.59,23.5868
